# Concrete Compressive Strength: EDA and Machine Learning Regression

**Portfolio context:** This project was designed as a clean demonstration notebook based on the MMIE 7310: Machine Learning and Artificial Intelligence for Engineers workflow. It combines exploratory data analysis, feature engineering, model development, model evaluation, and engineering interpretation for a nonlinear materials-property prediction problem.

## Problem Statement

Concrete compressive strength is a key mechanical property influenced by cement, supplementary cementitious materials, water, aggregates, admixtures, and curing age. The goal of this project is to build a reproducible machine learning workflow to predict compressive strength from mixture design variables.

## Dataset

The dataset is the **Concrete Compressive Strength Data Set** from the UCI Machine Learning Repository and is also available on Kaggle.

- Dataset source: https://archive.ics.uci.edu/ml/datasets/Concrete+Compressive+Strength
- Kaggle source may also be used if available.
- Target variable: `concrete_compressive_strength`
- Problem type: supervised regression

## 1. Import Libraries

The notebook uses standard Python data science libraries for tabular analysis, visualization, preprocessing, classical machine learning, hyperparameter tuning, and optional neural-network modeling.

In [ ]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.inspection import permutation_importance

from sklearn.linear_model import LinearRegression, Ridge
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

RANDOM_STATE = 42
pd.set_option("display.max_columns", 100)
sns.set_context("notebook")

## 2. Load Data

For GitHub sharing, the dataset is not included in this repository. Download the dataset from the source listed in the README and place it in the project root as `concrete_data.csv`.

This keeps the repository lightweight and respects dataset ownership/licensing.

In [ ]:
DATA_PATH = "concrete_data.csv"

df = pd.read_csv(DATA_PATH)

# Clean small formatting issues in column names
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
      .str.replace("-", "_")
)

df.head()

## 3. Dataset Understanding

This step identifies the number of observations, input variables, target variable, data types, and the engineering meaning of each feature. In engineering ML, this step is essential because prediction quality depends not only on algorithms but also on physically meaningful input variables.

In [ ]:
print(f"Number of samples: {df.shape[0]}")
print(f"Number of columns: {df.shape[1]}")

display(df.info())
display(df.head())

In [ ]:
target = "concrete_compressive_strength"
features = [col for col in df.columns if col != target]

feature_descriptions = pd.DataFrame({
    "Feature": features + [target],
    "Engineering meaning": [
        "Cement content in the concrete mixture",
        "Blast furnace slag content, a supplementary cementitious material",
        "Fly ash content, another supplementary cementitious material",
        "Water content affecting hydration and workability",
        "Superplasticizer dosage affecting flowability and water reduction",
        "Coarse aggregate content",
        "Fine aggregate content",
        "Curing age of the concrete specimen",
        "Measured compressive strength of concrete"
    ]
})
display(feature_descriptions)

## 4. Data Quality Assessment

Before modeling, we check missing values, duplicate records, and unrealistic values. In manufacturing or materials analytics, poor data quality can lead to misleading models and incorrect engineering conclusions.

In [ ]:
quality_summary = pd.DataFrame({
    "missing_values": df.isna().sum(),
    "missing_percent": (df.isna().mean() * 100).round(2),
    "unique_values": df.nunique(),
    "data_type": df.dtypes.astype(str)
})
display(quality_summary)

print(f"Duplicate rows: {df.duplicated().sum()}")

In [ ]:
# Remove duplicate records if present
df_clean = df.drop_duplicates().copy()
print(f"Shape before duplicate removal: {df.shape}")
print(f"Shape after duplicate removal: {df_clean.shape}")

## 5. Descriptive Statistics

Descriptive statistics provide a first look at central tendency, spread, and range. For concrete, these values also help confirm whether mixture proportions and strength values are physically reasonable.

In [ ]:
display(df_clean.describe().T)

## 6. Univariate Analysis

Histograms show distribution shape and skewness. Boxplots help identify potential outliers. Outliers in materials datasets may represent rare but valid mixture designs, so they should not be removed automatically without engineering justification.

In [ ]:
for col in df_clean.columns:
    plt.figure(figsize=(7, 4))
    sns.histplot(df_clean[col], kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.show()

In [ ]:
for col in df_clean.columns:
    plt.figure(figsize=(7, 2.8))
    sns.boxplot(x=df_clean[col])
    plt.title(f"Boxplot of {col}")
    plt.xlabel(col)
    plt.tight_layout()
    plt.show()

## 7. Bivariate Analysis

This section examines how each input variable relates to compressive strength. Scatter plots and correlation coefficients help identify important predictors and nonlinear patterns.

In [ ]:
for col in features:
    plt.figure(figsize=(6.5, 4))
    sns.scatterplot(data=df_clean, x=col, y=target, alpha=0.7)
    plt.title(f"{col} vs Concrete Compressive Strength")
    plt.xlabel(col)
    plt.ylabel("Compressive Strength")
    plt.tight_layout()
    plt.show()

In [ ]:
corr_with_target = df_clean.corr(numeric_only=True)[target].sort_values(ascending=False)
display(corr_with_target.to_frame("correlation_with_strength"))

## 8. Multivariate Analysis

A correlation heatmap helps identify multicollinearity and relationships among input features. Multicollinearity can affect linear models, while tree-based models are often more robust.

In [ ]:
plt.figure(figsize=(10, 8))
sns.heatmap(df_clean.corr(numeric_only=True), annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.show()

In [ ]:
selected_cols = ["cement", "water", "superplasticizer", "age", target]
sns.pairplot(df_clean[selected_cols], diag_kind="kde")
plt.suptitle("Pairplot of Selected Concrete Features", y=1.02)
plt.show()

## 9. Outlier Detection Using IQR

The IQR method flags values outside 1.5 times the interquartile range. These points are reviewed but retained for modeling unless there is evidence of data entry error.

In [ ]:
outlier_summary = []
for col in df_clean.columns:
    q1 = df_clean[col].quantile(0.25)
    q3 = df_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    n_outliers = ((df_clean[col] < lower) | (df_clean[col] > upper)).sum()
    outlier_summary.append([col, lower, upper, n_outliers])

outlier_df = pd.DataFrame(outlier_summary, columns=["feature", "lower_bound", "upper_bound", "n_outliers"])
display(outlier_df)

## 10. Feature Scaling and Normalization

Scaling is important for algorithms such as SVR, ridge regression, kNN, and neural networks. Tree-based models do not require scaling but are included in the same preprocessing framework for consistency.

In [ ]:
X = df_clean[features]
y = df_clean[target]

standard_scaler = StandardScaler()
minmax_scaler = MinMaxScaler()

X_standard = pd.DataFrame(standard_scaler.fit_transform(X), columns=features)
X_minmax = pd.DataFrame(minmax_scaler.fit_transform(X), columns=features)

display(X_standard.describe().T[["mean", "std", "min", "max"]])
display(X_minmax.describe().T[["mean", "std", "min", "max"]])

In [ ]:
plt.figure(figsize=(10, 5))
sns.boxplot(data=X_standard)
plt.title("Standardized Feature Distributions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

plt.figure(figsize=(10, 5))
sns.boxplot(data=X_minmax)
plt.title("Min-Max Normalized Feature Distributions")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 11. Train-Test Split

Scaling is fitted only on the training data and then applied to the test data. This prevents data leakage and reflects the correct machine learning workflow.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=RANDOM_STATE
)

print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)

## 12. Model Training and Evaluation

We compare multiple regression models:

1. Linear Regression  
2. Ridge Regression  
3. Decision Tree Regressor  
4. Random Forest Regressor  
5. Support Vector Regression  

The evaluation metrics are MAE, MSE, RMSE, and R².

In [ ]:
def evaluate_regression_model(name, model, X_train, X_test, y_train, y_test):
    model.fit(X_train, y_train)
    pred = model.predict(X_test)
    return {
        "model": name,
        "MAE": mean_absolute_error(y_test, pred),
        "MSE": mean_squared_error(y_test, pred),
        "RMSE": np.sqrt(mean_squared_error(y_test, pred)),
        "R2": r2_score(y_test, pred),
        "fitted_model": model,
        "predictions": pred
    }

models = {
    "Linear Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LinearRegression())
    ]),
    "Ridge Regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", Ridge(alpha=1.0))
    ]),
    "Decision Tree": DecisionTreeRegressor(max_depth=8, random_state=RANDOM_STATE),
    "Random Forest": RandomForestRegressor(n_estimators=300, random_state=RANDOM_STATE),
    "SVR RBF": Pipeline([
        ("scaler", StandardScaler()),
        ("model", SVR(kernel="rbf", C=100, gamma="scale"))
    ])
}

results = []
fitted_models = {}

for name, model in models.items():
    res = evaluate_regression_model(name, model, X_train, X_test, y_train, y_test)
    fitted_models[name] = res["fitted_model"]
    results.append({k: v for k, v in res.items() if k not in ["fitted_model", "predictions"]})

results_df = pd.DataFrame(results).sort_values("R2", ascending=False)
display(results_df)

## 13. Hyperparameter Tuning

We tune the random forest model because it often performs well for nonlinear tabular engineering data and provides interpretable feature importance.

In [ ]:
rf_pipeline = Pipeline([
    ("model", RandomForestRegressor(random_state=RANDOM_STATE))
])

param_grid = {
    "model__n_estimators": [100, 300],
    "model__max_depth": [None, 8, 12],
    "model__min_samples_leaf": [1, 2, 4]
}

grid = GridSearchCV(
    rf_pipeline,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best cross-validation R2:", grid.best_score_)

best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)

best_metrics = {
    "MAE": mean_absolute_error(y_test, best_pred),
    "MSE": mean_squared_error(y_test, best_pred),
    "RMSE": np.sqrt(mean_squared_error(y_test, best_pred)),
    "R2": r2_score(y_test, best_pred)
}
display(pd.DataFrame([best_metrics]))

## 14. Best Model Visualization

A predicted vs. actual plot shows how closely the model predictions match measured concrete strength. Points near the diagonal line indicate better model agreement.

In [ ]:
plt.figure(figsize=(6, 6))
sns.scatterplot(x=y_test, y=best_pred, alpha=0.8)
min_val = min(y_test.min(), best_pred.min())
max_val = max(y_test.max(), best_pred.max())
plt.plot([min_val, max_val], [min_val, max_val], linestyle="--")
plt.title("Predicted vs Actual Concrete Compressive Strength")
plt.xlabel("Actual Strength")
plt.ylabel("Predicted Strength")
plt.tight_layout()
plt.show()

## 15. Feature Importance

Feature importance helps translate model predictions into engineering insight. For concrete strength, curing age, cement content, water content, and admixture-related variables are often influential.

In [ ]:
rf = best_model.named_steps["model"]
importance_df = pd.DataFrame({
    "feature": features,
    "importance": rf.feature_importances_
}).sort_values("importance", ascending=False)

display(importance_df)

plt.figure(figsize=(8, 5))
sns.barplot(data=importance_df, x="importance", y="feature")
plt.title("Random Forest Feature Importance")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()

## 16. Optional ANN Regression Model

This section is optional. It demonstrates how a small feedforward neural network can be applied to the same regression problem. If TensorFlow is not installed, this cell can be skipped.

In [ ]:
try:
    import tensorflow as tf
    from tensorflow import keras
    from tensorflow.keras import layers

    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    ann = keras.Sequential([
        layers.Input(shape=(X_train_scaled.shape[1],)),
        layers.Dense(64, activation="relu"),
        layers.Dense(32, activation="relu"),
        layers.Dense(1)
    ])

    ann.compile(optimizer="adam", loss="mse", metrics=["mae"])

    history = ann.fit(
        X_train_scaled, y_train,
        validation_split=0.2,
        epochs=100,
        batch_size=32,
        verbose=0
    )

    ann_pred = ann.predict(X_test_scaled).ravel()
    print("ANN Test R2:", r2_score(y_test, ann_pred))
    print("ANN Test RMSE:", np.sqrt(mean_squared_error(y_test, ann_pred)))

    plt.figure(figsize=(7, 4))
    plt.plot(history.history["loss"], label="Training loss")
    plt.plot(history.history["val_loss"], label="Validation loss")
    plt.title("ANN Training Loss")
    plt.xlabel("Epoch")
    plt.ylabel("MSE Loss")
    plt.legend()
    plt.tight_layout()
    plt.show()

except ImportError:
    print("TensorFlow is not installed. Install tensorflow to run the ANN section.")

## 17. Engineering Interpretation and Conclusions

Key conclusions to document after running the notebook:

- Concrete strength is nonlinear and depends on both mixture composition and curing age.
- Scaling is essential for algorithms that depend on distance or gradient optimization.
- Tree-based models are strong baselines for nonlinear tabular engineering data.
- Feature importance provides engineering insight into which mixture variables drive strength predictions.
- The workflow is directly transferable to materials informatics, quality prediction, and manufacturing analytics.